# 🏗️ CEM4644 · MP4 — Segmentation for quantity take-off on floor plans
## Workshop (in class): *Residential floor plans, set A*

**No coding needed.** Each grey box below is one *step*: click the ▶ (play) button at its left, wait until it finishes, look at the result, then answer the report question that follows. Run the steps **from top to bottom**.

**What you will do (about 90 minutes)**
1. Look at real floor plans and what is drawn on them.
2. Ask a segmentation model, by name, for rooms, fixtures and openings: see the mask, the overlay, the count and the area, and compare with the drawing's own numbers.
3. Do a quantity take-off with boxes: check the scale, measure rooms in square metres, count windows and doors.
4. Compare two plans, and examine where the model goes wrong: wording, weak regions, and how to correct it.
5. Try a plan of your own.

**Before you start:** menu *Runtime → Change runtime type → T4 GPU → Save*. The model used here (SAM 3) is large: with a GPU each request takes well under a second; without one, the precomputed results still work but live requests take about a minute each.

In [ ]:
#@title ▶ Step 0 · Run me first (2–3 minutes) { display-mode: "form" }
#@markdown Click ▶ and wait for the green ✅ line. This downloads the plans with their precomputed results and loads SAM 3 (about 3 GB).
#@markdown Untick *load_model* only if you have no GPU and want to skip the live steps.
load_model = True #@param {type:"boolean"}
import importlib, os, shutil, subprocess, sys
REPO, FOLDER, PKG = "CEM4644", "mp4_segmentation", "aec_seg"

def _git(*args):
    return subprocess.run(["git", "-C", REPO, *args], capture_output=True, text=True).returncode == 0

if os.path.isdir(REPO):                      # a copy is already here: pull the newest course code over it
    if not (_git("fetch", "-q", "--depth", "1", "origin", "master")
            and _git("reset", "-q", "--hard", "FETCH_HEAD") and _git("clean", "-qfd")):
        shutil.rmtree(REPO, ignore_errors=True)          # broken copy: start again from scratch
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "-q", "https://github.com/Haolan-Zhang/CEM4644.git", REPO], check=True)
for _m in [m for m in list(sys.modules) if m == PKG or m.startswith(PKG + ".")]:
    del sys.modules[_m]                      # Python caches imported code: drop it, or this cell keeps the old version
importlib.invalidate_caches()
sys.path.insert(0, os.path.abspath(os.path.join(REPO, FOLDER)))
from aec_seg import lab
lab.setup(dataset="homes_a", load_model=load_model)


## Part 1 · Meet the plans

Detection (MP3) draws a **box** around an object. **Segmentation** goes one step further: it decides, *pixel by pixel*, what belongs to the object. On a drawing that is what makes it useful for **quantity take-off**: count the pixels of a room and you have its area in pixels; know the scale and you have square metres.

The model is **SAM 3** (Segment Anything Model 3, Meta 2025). You do not train it. You type a short phrase, such as *room* or *toilet*, and it returns every region of the drawing that matches, each with a **confidence**. You can also draw a **box** around something: SAM 3 cuts out its outline, and can look for everything else that looks like it.

Six real architectural floor plans of Finnish homes from the CubiCasa5K dataset (CC BY-NC-SA 4.0), redrawn at a known scale with a 5 m scale bar. Each plan comes with an answer key (every room's real area, every door, window and fixture) that the notebook uses to check your measurements.

Every plan has a **5 m scale bar** at the bottom left. Room labels are abbreviations in Finnish (one plan is Swedish):

| label | meaning |
|---|---|
| OH | olohuone = living room |
| MH | makuuhuone = bedroom |
| K / KT / KEITTIÖ | keittiö = kitchen |
| KH / KPH | kylpyhuone = bathroom |
| WC | toilet |
| S | sauna |
| ET | eteinen = entrance hall |
| VH | vaatehuone = walk-in closet |
| PH | pesuhuone = washroom |
| KHH | kodinhoitohuone = utility room |
| VAR | varasto = storage |
| TK | tekninen tila = technical room |
| PARVEKE / PARV | balcony |
| TERASSI | terrace |
| AT / AUTOTALLI | garage |
| RT | ruokailutila = dining area |
| TUPA | farmhouse living room |
| SOVR / KÖK / BAD / HALL | Swedish: bedroom / kitchen / bathroom / hall |
| m² | square metres (printed on some plans) |

In [ ]:
#@title ▶ Step 1a · Browse the plans { display-mode: "form" }
#@markdown *all plans* shows every plan with what the drawing contains (rooms, floor area, doors, windows). These facts come from the plans' own annotations and are the answer key the notebook checks you against.
which = "all plans" #@param ["all plans", "13828: detached house with furniture, 8 rooms", "9493: detached house with garage and terrace", "14466: apartment with balcony (bold walls)", "11032: detached house with terrace", "548: apartment next to a stair core", "1902: studio flat, 24 m² (bold walls)"]
lab.show_plans(which)


## Part 2 · Ask for something by name

Pick a plan and a thing. You get three panels: the drawing, the **mask** (white = the model says *this is it*), and the **overlay**. Below them: how many regions, how many pixels, how many square metres (using the plan's scale), and what the **drawing's own answer key** says. The **confidence slider** hides the regions the model is unsure about: watch the count and the area change.

In [ ]:
#@title ▶ Step 2a · Original → mask → overlay { display-mode: "form" }
#@markdown Try *room (any)*, then *bedroom* and *bathroom*; then the symbols *toilet*, *sink*, *stairs*; then *kitchen*, *door* and *window*. Some words work, some find nothing at all: that is part of the lesson.
plan = "13828: detached house with furniture, 8 rooms" #@param ["13828: detached house with furniture, 8 rooms", "9493: detached house with garage and terrace", "14466: apartment with balcony (bold walls)", "11032: detached house with terrace", "548: apartment next to a stair core", "1902: studio flat, 24 m² (bold walls)"]
thing = "room (any)" #@param ["room (any)", "bedroom", "bathroom", "kitchen", "living room", "balcony / terrace", "toilet", "sink", "bathtub", "stairs", "door", "window", "wall"]
confidence = 0.3 #@param {type:"slider", min:0.1, max:0.9, step:0.05}
lab.segment(plan, thing, confidence)


In [ ]:
#@title ▶ Step 2b · Hits, misses and extras { display-mode: "form" }
#@markdown The answer key drawn on the plan: green = a real one the model found, red = a real one it missed, blue = a region that is not one.
plan = "13828: detached house with furniture, 8 rooms" #@param ["13828: detached house with furniture, 8 rooms", "9493: detached house with garage and terrace", "14466: apartment with balcony (bold walls)", "11032: detached house with terrace", "548: apartment next to a stair core", "1902: studio flat, 24 m² (bold walls)"]
thing = "window" #@param ["room (any)", "bedroom", "bathroom", "kitchen", "living room", "balcony / terrace", "toilet", "sink", "bathtub", "stairs", "door", "window", "wall"]
confidence = 0.3 #@param {type:"slider", min:0.1, max:0.9, step:0.05}
lab.count(plan, thing, confidence)


In [ ]:
#@title ▶ Step 2c · Every room type at once { display-mode: "form" }
#@markdown SAM 3's square metres per room type next to the drawing's, and the counts of fixtures and openings next to the truth.
plan = "13828: detached house with furniture, 8 rooms" #@param ["13828: detached house with furniture, 8 rooms", "9493: detached house with garage and terrace", "14466: apartment with balcony (bold walls)", "11032: detached house with terrace", "548: apartment next to a stair core", "1902: studio flat, 24 m² (bold walls)"]
confidence = 0.3 #@param {type:"slider", min:0.1, max:0.9, step:0.05}
lab.mix(plan, confidence)


In [ ]:
#@title ▶ Step 2d · Can *you* estimate an area? { display-mode: "form" }
#@markdown A room is outlined on a plan: guess its area from the scale bar, then see the drawing's number and SAM 3's.
rounds = 4 #@param {type:"slider", min:2, max:8, step:1}
lab.guess_game(rounds)


> ### 📝 Report question 1
> From Step 2a and 2b: which words found what they should (rooms? toilets? windows? doors?), and which found nothing or something else? Give the found / missed / extra counts for two things on one plan at confidence 0.3, and say what the misses have in common.

> ### 📝 Report question 2
> From Step 2c: for one plan, copy the table of square metres per room type (SAM 3 vs the drawing). Which room type is measured best and which worst, and why (merged rooms, furniture, open-plan kitchen and living room)? Then move the confidence to 0.2 and 0.7: what changes? Also give your score in the estimation game.

## Part 3 · Quantity take-off with boxes

A phrase is quick, but a take-off needs control. So now you draw the boxes. Three steps: **check the scale** on the scale bar (a 1 % error in the scale is a 2 % error in every area), **measure** rooms and elements by drawing a tight box around each (SAM 3 cuts out the outline inside the box; the notebook compares the area with the drawing), and **count** with *find_all*: draw one box around a window, and SAM 3 looks for every other thing that looks like it. The drawing's answer key tells you what it found, missed and added.

In [ ]:
#@title ▶ Step 3a · Check the scale { display-mode: "form" }
#@markdown Draw a box from the 0 tick to the 5 m tick of the scale bar (zoom in with the mouse wheel for precision), label it *scale bar*, click *Submit*.
plan = "13828: detached house with furniture, 8 rooms" #@param ["13828: detached house with furniture, 8 rooms", "9493: detached house with garage and terrace", "14466: apartment with balcony (bold walls)", "11032: detached house with terrace", "548: apartment next to a stair core", "1902: studio flat, 24 m² (bold walls)"]
lab.scale_check(plan)


In [ ]:
#@title ▶ Step 3b · Measure and count with boxes { display-mode: "form" }
#@markdown Draw tight boxes: label each one *room*, *window*, *door*, *fixture* or *other*, then *Submit*. Measure at least two rooms of your choice, one window and one door. With *find_all* ticked, SAM 3 also looks for everything like your first window / door / fixture / room box (blue boxes; thin green = the drawing's answer key). Needs the live model.
plan = "13828: detached house with furniture, 8 rooms" #@param ["13828: detached house with furniture, 8 rooms", "9493: detached house with garage and terrace", "14466: apartment with balcony (bold walls)", "11032: detached house with terrace", "548: apartment next to a stair core", "1902: studio flat, 24 m² (bold walls)"]
find_all = True #@param {type:"boolean"}
confidence = 0.3 #@param {type:"slider", min:0.1, max:0.9, step:0.05}
lab.takeoff(plan, find_all, confidence)


> ### 📝 Report question 3
> From Step 3b: which rooms did you measure, what did you get, and what does the drawing say? Give the error in percent for each and explain where it comes from (your box, the mask stopping at furniture or a door opening, the scale).

> ### 📝 Report question 4
> From Step 3b with *find_all* on a window: how many windows does the drawing have, how many did SAM 3 find, how many did it miss and how many were extra? Which confidence worked best, and what did the extras have in common?

## Part 4 · Compare plans, and where it goes wrong

Two plans side by side: square metres per room type from SAM 3, and the drawings' own totals. Then three kinds of error to look for: the **words** you use (the model was trained on everyday photos, not on drawings), **weak regions** it proposes with low confidence, and plain **mistakes** that need a correction. The last step lets you correct the model by drawing a box over what it got wrong.

In [ ]:
#@title ▶ Step 4a · Compare two plans { display-mode: "form" }
plan_a = "13828: detached house with furniture, 8 rooms" #@param ["13828: detached house with furniture, 8 rooms", "9493: detached house with garage and terrace", "14466: apartment with balcony (bold walls)", "11032: detached house with terrace", "548: apartment next to a stair core", "1902: studio flat, 24 m² (bold walls)"]
plan_b = "9493: detached house with garage and terrace" #@param ["13828: detached house with furniture, 8 rooms", "9493: detached house with garage and terrace", "14466: apartment with balcony (bold walls)", "11032: detached house with terrace", "548: apartment next to a stair core", "1902: studio flat, 24 m² (bold walls)"]
confidence = 0.3 #@param {type:"slider", min:0.1, max:0.9, step:0.05}
lab.compare(plan_a, plan_b, confidence)


In [ ]:
#@title ▶ Step 4b · Does the wording matter? { display-mode: "form" }
#@markdown The same thing asked for with different words; all wordings are precomputed.
plan = "13828: detached house with furniture, 8 rooms" #@param ["13828: detached house with furniture, 8 rooms", "9493: detached house with garage and terrace", "14466: apartment with balcony (bold walls)", "11032: detached house with terrace", "548: apartment next to a stair core", "1902: studio flat, 24 m² (bold walls)"]
thing = "bathroom" #@param ["room (any)", "bedroom", "bathroom", "kitchen", "living room", "balcony / terrace", "toilet", "sink", "bathtub", "stairs", "door", "window", "wall"]
confidence = 0.3 #@param {type:"slider", min:0.1, max:0.9, step:0.05}
lab.phrase_lab(plan, thing, confidence)


In [ ]:
#@title ▶ Step 4c · Look at each region and its confidence { display-mode: "form" }
#@markdown Every region the model proposed, numbered, with its confidence, its area and the room it sits on. Move the slider to see which ones survive.
plan = "13828: detached house with furniture, 8 rooms" #@param ["13828: detached house with furniture, 8 rooms", "9493: detached house with garage and terrace", "14466: apartment with balcony (bold walls)", "11032: detached house with terrace", "548: apartment next to a stair core", "1902: studio flat, 24 m² (bold walls)"]
thing = "room (any)" #@param ["room (any)", "bedroom", "bathroom", "kitchen", "living room", "balcony / terrace", "toilet", "sink", "bathtub", "stairs", "door", "window", "wall"]
lab.inspect(plan, thing)


In [ ]:
#@title ▶ Step 4d · Correct it with a box { display-mode: "form" }
#@markdown Draw a box over a region that is wrong, click *Submit*: SAM 3 runs again with your box as a *not this* hint. Needs the live model.
plan = "13828: detached house with furniture, 8 rooms" #@param ["13828: detached house with furniture, 8 rooms", "9493: detached house with garage and terrace", "14466: apartment with balcony (bold walls)", "11032: detached house with terrace", "548: apartment next to a stair core", "1902: studio flat, 24 m² (bold walls)"]
thing = "kitchen" #@param ["room (any)", "bedroom", "bathroom", "kitchen", "living room", "balcony / terrace", "toilet", "sink", "bathtub", "stairs", "door", "window", "wall"]
confidence = 0.3 #@param {type:"slider", min:0.1, max:0.9, step:0.05}
lab.fix(plan, thing, confidence)


In [ ]:
#@title ▶ Step 4e · Your own words { display-mode: "form" }
#@markdown Type any phrase: a room, a symbol, a shape (*small square*, *circle*, *thick line*). Needs the live model.
plan = "13828: detached house with furniture, 8 rooms" #@param ["13828: detached house with furniture, 8 rooms", "9493: detached house with garage and terrace", "14466: apartment with balcony (bold walls)", "11032: detached house with terrace", "548: apartment next to a stair core", "1902: studio flat, 24 m² (bold walls)"]
phrase = "bed" #@param {type:"string"}
confidence = 0.3 #@param {type:"slider", min:0.1, max:0.9, step:0.05}
lab.your_phrase(plan, phrase, confidence)


> ### 📝 Report question 5
> From Step 4a: which plan has more bedroom area and more bathroom area according to SAM 3, and does the drawing agree? From Step 4b: which wording worked best for the thing you chose, and how different were the areas?

> ### 📝 Report question 6
> Describe one mistake you found in Step 4c or 4d (what was included or missed, at which confidence). Did the negative box fix it? What would you tell a colleague who wants to use these square metres in a cost estimate?

## Part 5 · Your own plan

In [ ]:
#@title ▶ Your plan, your words { display-mode: "form" }
#@markdown Upload a floor plan (a photo of a drawing works too, or open the public link on your phone), type what to find, move the threshold. If the plan has a scale bar, measure how many pixels one metre is and enter the centimetres per pixel to get square metres.
#@markdown Test at least 1 plan(s) of your own and take screenshots for your report. Needs the live model.
lab.upload_app()


> ### 📝 Report question 7
> Test 1 plan(s) of your own (any floor plan from the internet or a course). For each: the phrase you used, the count and area measured, and whether the mask is right. What kind of drawing or wording failed?

> ### 📝 Report question 8
> Where in a project would a take-off like this be useful, and where would it mislead? What would you need (clean drawings, a scale, a room schedule, a person checking) to turn it into numbers you would put in an estimate?

## Wrap-up

In [ ]:
#@title ▶ Numbers for your report { display-mode: "form" }
lab.report_summary()


### Plan credits and model
- Floor plans: CubiCasa5K (Kalervo, Ylioinas, Häikiö, Karhu, Kannala 2019), CubiCasa Oy, licence CC BY-NC-SA 4.0, https://zenodo.org/records/2613548. Sample ids in this notebook: 13828, 9493, 14466, 11032, 548, 1902 (folder `data/plans/homes_a`, credits in `credits.json`). The plans were resampled to a plan-specific scale and given a scale bar; the answer keys come from the dataset's vector annotations.
- Model: SAM 3 by Meta AI (SAM License), loaded from a public mirror of the official checkpoint; a copy of the licence is in `docs/SAM_LICENSE.txt`.
- Lab code: https://github.com/Haolan-Zhang/CEM4644 (folder `mp4_segmentation`).